# Demo: end-to-end context inference

Required submission deliverable: one end-to-end inference example.
Raw clip -> mel/chroma -> segment graph, raw tags -> pseudo-caption -> BERT
embedding, fusion -> predicted tags (+ valence/arousal if a Task 3 model with
`predict_emotion=True` is loaded).

In [1]:
import sys, os, torch
sys.path.append('..')
from src.utils import load_config, load_checkpoint
from src.audio_features import process_clip
from src.graph_builder import build_segment_graph
from src.dataset import load_magnatagatune, tags_to_pseudo_caption, _resolve_path
from src.fusion_model import GnnBertFusion

cfg = load_config('../config.yaml')
device = 'cpu'


In [2]:
# 1. Pick one MagnaTagATune clip
mtt = load_magnatagatune(cfg)
tag_vocab = mtt.attrs['tag_vocab']
row = mtt.iloc[0]
audio_dir = _resolve_path(cfg['data']['magnatagatune']['audio_dir'])
audio_path = f"{audio_dir}/{row['mp3_path']}"
true_tags = [t for t in tag_vocab if row[t] == 1]
print('true tags:', true_tags)


true tags: ['classical', 'strings', 'violin', 'opera']


In [3]:
# 2. Audio -> mel/chroma -> segment graph
feats = process_clip(audio_path, cfg)
graph = build_segment_graph(feats['segment_features'], cfg['graph']['segment_similarity_threshold']).to(device)
graph.batch = torch.zeros(graph.x.size(0), dtype=torch.long, device=device)

# 3. Tags -> pseudo-caption -> tokens
caption = tags_to_pseudo_caption(true_tags)
print('pseudo-caption:', caption)

/Users/sifatshariar/Downloads/gnn-bert-music-context/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


pseudo-caption: a classical, opera track featuring strings, violin


In [4]:
# 4. Load a trained Task 3 fusion checkpoint and run inference
model = GnnBertFusion(
    graph_in_dim=graph.x.shape[1], num_tags=len(tag_vocab),
    bert_model_name=cfg['text']['bert_model'],
    gnn_hidden_dim=cfg['model']['gnn_hidden_dim'], gnn_layers=cfg['model']['gnn_num_layers'],
    gnn_dropout=cfg['model']['gnn_dropout'], gnn_type=cfg['model']['gnn_type'],
    attn_heads=cfg['model']['attention_heads'], fusion_dim=cfg['model']['fusion_dim'],
    fine_tune_bert=cfg['text']['fine_tune_bert'], fusion_type='cross_attention',
).to(device)

ckpt_path = _resolve_path('results/task3_cross_attention_best.pt')
load_checkpoint(ckpt_path, model, map_location=device, strict=False)
model.eval()

enc = model.text_encoder.tokenizer([caption], return_tensors='pt', padding=True, truncation=True,
                                    max_length=cfg['text']['max_length']).to(device)
with torch.no_grad():
    out = model(enc['input_ids'], enc['attention_mask'], graph.x, graph.edge_index, graph.batch)
    probs = out['tag_logits'].sigmoid().cpu().numpy()[0]

top5 = sorted(zip(tag_vocab, probs), key=lambda x: -x[1])[:5]
print('predicted top-5 tags:', top5)
print('true tags:', true_tags)


predicted top-5 tags: [('opera', np.float32(0.6578007)), ('classical', np.float32(0.37746775)), ('female', np.float32(0.16177487)), ('choir', np.float32(0.1585013)), ('male', np.float32(0.12178367))]
true tags: ['classical', 'strings', 'violin', 'opera']
